In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/"
gold_adls_path = "abfss://gold@"+storage_account+".dfs.core.windows.net/"

In [0]:
claims = spark.read.format("delta").load(silver_adls_path+"/claims")
enrollment = spark.read.format("delta").load(silver_adls_path+"/customer_enrollment")
policies = spark.read.format("delta").load(silver_adls_path+"/policy_products")
hospitals = spark.read.format("delta").load(silver_adls_path+"/hospitals")

# hospitals.printSchema()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

fact_claims = spark.read.format("delta").load(gold_adls_path+"/fact_claims")

agg_claim_summary = (
    fact_claims
    .groupBy(
        "claim_year",
        "claim_month",
        "region"
    )
    .agg(
        count("claim_id").alias("total_claims"),
        sum("claimed_amount").alias("total_claimed_amount"),
        sum("approved_amount").alias("total_approved_amount"),
        avg("approval_rate").alias("avg_approval_rate"),
        avg("length_of_stay").alias("avg_hospital_stay")
    )
)

agg_claim_summary.write.format("delta").mode("overwrite").save(gold_adls_path+"/agg_claims_summary")